# LeetCode 2880 — Select Data bằng Python (pandas)

Link đề: https://leetcode.com/problems/select-data/ · Độ khó: **Easy** · Chủ đề: Pandas (30 Days of Pandas)

Skeleton LeetCode (không có `class Solution`; tên hàm / chữ ký: đối chiếu lại với đề gốc):

```python
import pandas as pd

def selectData(students: pd.DataFrame) -> pd.DataFrame:
    ...
```

Ghi chú: dữ liệu ví dụ trong notebook được viết theo trí nhớ về đề; nên đối chiếu lại với đề gốc nếu muốn khớp từng ô.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: từ bảng `students`, **lọc hàng** có `student_id = 101` rồi **chỉ giữ hai cột** `name` và `age`.
2. Nắm khái niệm **boolean mask** (cột `True/False`) để lọc hàng và **danh sách tên cột** để chọn cột; biết gộp hai việc bằng `df.loc[mask, [cột]]`.
3. Đi từ cách **vòng lặp thuần Python** → **lọc hai bước `df[mask][cột]`** → **`loc[mask, cột]`** → **`query`** → **`iloc` với chỉ số vị trí**.
4. Phân biệt `df["a"]` (trả về **Series**) với `df[["a"]]` (trả về **DataFrame**), `==` với `=`, `&` với `and`.
5. Tránh các lỗi kinh điển: `=` thay `==`, `and` thay `&`, `df["name", "age"]` (thiếu ngoặc vuông), quên chọn cột, đổi dữ liệu gốc thay vì tạo kết quả mới.

## 2. Đề bài & ví dụ

Cho DataFrame `students` với các cột `student_id`, `name`, `age`.

Hãy viết hàm **chọn các hàng có `student_id = 101`** và **chỉ hiển thị hai cột `name` và `age`**.

**Ví dụ**

- Input: DataFrame `students`

| student_id | name | age |
|---|---|---|
| 101 | Ulysses | 13 |
| 53 | William | 10 |
| 128 | Henry | 6 |
| 3 | Henry | 11 |

- Output:

| name | age |
|---|---|
| Ulysses | 13 |

- Giải thích: chỉ học sinh có `student_id = 101` được giữ lại; chỉ hai cột `name`, `age` được hiển thị (cột `student_id` bị bỏ).

(Giá trị cụ thể trong bảng: đối chiếu lại với đề gốc.)

In [1]:
from typing import List
import pandas as pd
import numpy as np

students = pd.DataFrame({
    "student_id": [101, 53, 128, 3],
    "name": ["Ulysses", "William", "Henry", "Henry"],
    "age": [13, 10, 6, 11],
})
print(students)
print(students.dtypes)

   student_id     name  age
0         101  Ulysses   13
1          53  William   10
2         128    Henry    6
3           3    Henry   11
student_id    int64
name            str
age           int64
dtype: object


## 3. Kiến thức nền

### 3.1. Chọn một cột và chọn nhiều cột

- `df["name"]` (một **chuỗi**) → trả về **Series** (một cột, có index).
- `df[["name", "age"]]` (một **list** tên cột) → trả về **DataFrame** gồm các cột đó, đúng thứ tự trong list.

Hai ngoặc vuông ở `[["name", "age"]]`: ngoặc ngoài là "chọn", ngoặc trong là **list** tên cột.

In [2]:
print(type(students["name"]).__name__)          # Series
print(type(students[["name"]]).__name__)        # DataFrame (một cột nhưng vẫn là bảng)
print(students[["name", "age"]])
print(students[["age", "name"]].columns.tolist())   # thứ tự theo list ta đưa

Series
DataFrame
      name  age
0  Ulysses   13
1  William   10
2    Henry    6
3    Henry   11
['age', 'name']


### 3.2. Phép so sánh trên cả cột → **boolean mask**

`students["student_id"] == 101` so sánh **từng phần tử** của cột với `101`, trả về một Series `True/False` cùng độ dài. Đó là **mặt nạ (mask)**: `True` = giữ hàng, `False` = bỏ hàng.

In [3]:
mask = students["student_id"] == 101
print(mask)
print(mask.tolist(), mask.sum(), "hàng thỏa điều kiện")

0     True
1    False
2    False
3    False
Name: student_id, dtype: bool
[True, False, False, False] 1 hàng thỏa điều kiện


### 3.3. Lọc hàng bằng mask: `df[mask]`

Đưa mask vào dấu `[]` của DataFrame: giữ các hàng có `True`. **Nhãn hàng (index) được giữ nguyên**, cột giữ nguyên.

In [4]:
print(students[mask])                  # chỉ hàng của Ulysses (nhãn 0), đủ 3 cột
print(students[students["age"] > 10])  # age > 10: Ulysses (13) và nhãn 3 (11)

   student_id     name  age
0         101  Ulysses   13
   student_id     name  age
0         101  Ulysses   13
3           3    Henry   11


### 3.4. `df.loc[hàng, cột]`

`loc` chọn đồng thời theo **nhãn hàng** và **nhãn cột**: `df.loc[điều_kiện_hàng, [danh_sách_cột]]`. Điều kiện hàng có thể là mask. Dùng `loc` một lần thay cho hai bước lọc.

In [5]:
print(students.loc[mask, ["name", "age"]])
print(students.loc[mask, "name"])          # cột là CHUỖI -> trả Series
print(students.loc[mask, ["name"]])        # cột là LIST -> trả DataFrame

      name  age
0  Ulysses   13
0    Ulysses
Name: name, dtype: str
      name
0  Ulysses


### 3.5. Kết hợp điều kiện: `&`, `|`, `~` (nhớ ngoặc đơn!)

Với Series, dùng `&` (và), `|` (hoặc), `~` (phủ định) — **không** dùng `and`, `or`, `not`. Mỗi vế phải bọc ngoặc đơn vì `&` có độ ưu tiên cao hơn `==`.

In [6]:
m = (students["student_id"] == 101) | (students["name"] == "Henry")
print(m.tolist())                                    # True, False, True, True
print(students[(students["name"] == "Henry") & (students["age"] > 8)])    # chỉ Henry 11 tuổi
print((~(students["name"] == "Henry")).tolist())     # phủ định

[True, False, True, True]
   student_id   name  age
3           3  Henry   11
[True, True, False, False]


### 3.6. `isin`, `eq` và `query`

- `df["x"].isin([a, b])` → mask "x thuộc danh sách".
- `df["x"].eq(a)` = `df["x"] == a` (dạng phương thức, nối chuỗi dễ hơn).
- `df.query("student_id == 101")` nhận **biểu thức dưới dạng chuỗi**; dùng `@biến` để lấy biến Python.

In [7]:
print(students[students["student_id"].isin([101, 3])].shape)    # (2, 3)
print(students.query("student_id == 101").shape)                # (1, 3)
sid = 128
print(students.query("student_id == @sid")["name"].tolist())    # ['Henry']

(2, 3)
(1, 3)
['Henry']


### 3.7. `iloc` và chỉ số vị trí số nguyên

`df.iloc[danh_sách_vị_trí_hàng, danh_sách_vị_trí_cột]` chọn theo **vị trí** (bắt đầu từ 0). `np.flatnonzero(mask)` cho vị trí các hàng `True`; `df.columns.get_indexer([...])` đổi tên cột thành vị trí.

In [8]:
pos_rows = np.flatnonzero(mask.to_numpy())
pos_cols = students.columns.get_indexer(["name", "age"])
print(pos_rows, pos_cols)                          # [0] [1 2]
print(students.iloc[pos_rows, pos_cols])

[0] [1 2]
      name  age
0  Ulysses   13


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Hai việc **độc lập**:

1. **Lọc hàng** (theo chiều dọc của bảng): đi qua từng hàng, giữ hàng nếu `student_id == 101`.
2. **Chọn cột** (theo chiều ngang): từ hàng còn lại, chỉ lấy `name` và `age`.

```
student_id name     age        giữ?
   101     Ulysses  13         ✓  (student_id == 101)
    53     William  10         ✗
   128     Henry     6         ✗
     3     Henry    11         ✗
```

Sau khi lọc còn một hàng; chỉ giữ hai cột nên kết quả là bảng 1 hàng × 2 cột: `Ulysses | 13`. Nhãn hàng vẫn là `0` (nhãn gốc của hàng đó), cột `student_id` đã bị bỏ.

Mẹo nghĩ: **hàng = điều kiện**, **cột = danh sách tên**; viết đúng thứ tự `loc[điều_kiện, danh_sách_cột]`.

## 5. Cách 1 — Brute Force / baseline: vòng lặp thuần Python

Duyệt từng hàng. Nếu `student_id == 101` thì lấy `name`, `age` đưa vào list; giữ lại **nhãn gốc** của hàng; cuối cùng dựng DataFrame với cột `["name", "age"]`.

In [9]:
def select_loop(students: pd.DataFrame) -> pd.DataFrame:
    labels, rows = [], []
    for label, sid, name, age in zip(students.index, students["student_id"], students["name"], students["age"]):
        if sid == 101:
            labels.append(label)
            rows.append((name, age))
    return pd.DataFrame(rows, index=labels, columns=["name", "age"])

out = select_loop(students)
print(out)
print(out.dtypes.tolist())

      name  age
0  Ulysses   13
[<StringDtype(storage='python', na_value=nan)>, dtype('int64')]


**Độ phức tạp:** duyệt `m` hàng, mỗi hàng `O(1)`: **Time `O(m)`** (nhưng trong **vòng lặp Python**, chậm hơn nhiều so với phép vector hóa), **Space `O(r)`** với `r` là số hàng khớp.

**Nút thắt:** vòng lặp Python chậm và dài; phải tự lo nhãn, dtype. pandas cho phép tính điều kiện **cho cả cột trong một lần** (vector hóa).

## 6. Ý tưởng thuật toán / bước đột phá

**Vector hóa:** thay vì hỏi từng hàng "bạn có `student_id == 101` không?", ta hỏi **cả cột** một lần:

`mask = students["student_id"] == 101`

rồi dùng mask làm công cụ lọc. Ba nấc từ thô đến gọn:

1. `students[mask][["name", "age"]]` — hai bước (lọc hàng rồi chọn cột);
2. `students.loc[mask, ["name", "age"]]` — **một bước**, rõ ràng nhất;
3. `students.query("student_id == 101")[["name", "age"]]` — viết điều kiện bằng chuỗi.

Cũng có thể làm theo vị trí số nguyên (`iloc`). Cả bốn cùng cho **cùng một kết quả**.

## 7. Các cách tiếp theo

### Cách 2 — Lọc hai bước `df[mask][["name", "age"]]`

**Giải thích từng đoạn.**

- `students["student_id"] == 101` → mask.
- `students[mask]` → bảng chỉ các hàng khớp (còn đủ 3 cột).
- `[["name", "age"]]` → chọn tiếp hai cột.

Dễ đọc; nhược điểm nhỏ là tạo một bảng trung gian (đủ cột) trước khi bỏ bớt cột.

In [10]:
def select_two_steps(students: pd.DataFrame) -> pd.DataFrame:
    filtered = students[students["student_id"] == 101]
    return filtered[["name", "age"]]

print(select_two_steps(students))

      name  age
0  Ulysses   13


In [11]:
# Code nộp LeetCode cho Cách 2
import pandas as pd

def selectData(students: pd.DataFrame) -> pd.DataFrame:
    return students[students["student_id"] == 101][["name", "age"]]

print(selectData(students))

      name  age
0  Ulysses   13


**Độ phức tạp Cách 2:** tạo mask `O(m)`, lọc `O(m)` (vector hóa, trong C), chọn cột `O(r)`. **Time `O(m)`**, **Space `O(m + r·c)`** (mask và bảng trung gian).

---

### Cách 3 — `loc[mask, cột]` (cách chuẩn)

**Giải thích.** `loc` nhận **hai** đối số: điều kiện hàng và danh sách cột. Một lần gọi, không có bảng trung gian đủ cột.

In [12]:
def select_loc(students: pd.DataFrame) -> pd.DataFrame:
    return students.loc[students["student_id"] == 101, ["name", "age"]]

print(select_loc(students))

      name  age
0  Ulysses   13


In [13]:
# Code nộp LeetCode cho Cách 3
import pandas as pd

def selectData(students: pd.DataFrame) -> pd.DataFrame:
    return students.loc[students["student_id"] == 101, ["name", "age"]]

print(selectData(students))

      name  age
0  Ulysses   13


**Độ phức tạp Cách 3:** **Time `O(m)`**, **Space `O(m + r·c)`** (mask `m` phần tử bool + kết quả).

---

### Cách 4 — `query`

**Giải thích.** `query("student_id == 101")` lọc theo biểu thức chuỗi (tên cột dùng trực tiếp). Sau đó chọn cột. Rất ngắn khi điều kiện phức tạp: `query("age > 10 and name == 'Henry'")` (trong `query` **được** dùng `and`).

In [14]:
def select_query(students: pd.DataFrame) -> pd.DataFrame:
    return students.query("student_id == 101")[["name", "age"]]

print(select_query(students))
print(students.query("age > 10 and name == 'Henry'")[["name", "age"]])

      name  age
0  Ulysses   13
    name  age
3  Henry   11


In [15]:
# Code nộp LeetCode cho Cách 4
import pandas as pd

def selectData(students: pd.DataFrame) -> pd.DataFrame:
    return students.query("student_id == 101")[["name", "age"]]

print(selectData(students))

      name  age
0  Ulysses   13


**Độ phức tạp Cách 4:** như Cách 3; `query` có thêm một chút chi phí phân tích chuỗi (hằng số).

---

### Cách 5 — `iloc` với chỉ số vị trí

**Giải thích.** Tìm **vị trí** các hàng khớp bằng `np.flatnonzero`, **vị trí** hai cột bằng `get_indexer`, rồi `iloc[vị_trí_hàng, vị_trí_cột]`. Dài hơn, nhưng làm rõ khác biệt `loc` (nhãn) và `iloc` (vị trí); hữu ích khi tên cột có thể trùng hoặc làm việc với mảng.

In [16]:
def select_iloc(students: pd.DataFrame) -> pd.DataFrame:
    rows = np.flatnonzero((students["student_id"] == 101).to_numpy())
    cols = students.columns.get_indexer(["name", "age"])
    return students.iloc[rows, cols]

print(select_iloc(students))

      name  age
0  Ulysses   13


**Độ phức tạp Cách 5:** **Time `O(m)`**, **Space `O(m + r·c)`**.

## 8. TRACE từng bước

In DataFrame **sau mỗi bước biến đổi** của Cách 2 / Cách 3: bảng gốc → mask → bảng sau khi lọc hàng → bảng sau khi chọn cột.

In [17]:
def trace_select(students):
    print("BƯỚC 0 - bảng gốc", students.shape)
    print(students)

    mask = students["student_id"] == 101
    print("\nBƯỚC 1 - mask = students['student_id'] == 101")
    print(pd.DataFrame({"student_id": students["student_id"], "mask": mask}))

    filtered = students[mask]
    print("\nBƯỚC 2 - students[mask]  (lọc hàng, giữ đủ cột)", filtered.shape)
    print(filtered)

    result = filtered[["name", "age"]]
    print("\nBƯỚC 3 - [['name', 'age']]  (chọn cột)", result.shape)
    print(result)

    print("\nSo với loc một bước:")
    direct = students.loc[mask, ["name", "age"]]
    print(direct)
    print("bằng nhau?", direct.equals(result))
    return result

_ = trace_select(students)

BƯỚC 0 - bảng gốc (4, 3)
   student_id     name  age
0         101  Ulysses   13
1          53  William   10
2         128    Henry    6
3           3    Henry   11

BƯỚC 1 - mask = students['student_id'] == 101
   student_id   mask
0         101   True
1          53  False
2         128  False
3           3  False

BƯỚC 2 - students[mask]  (lọc hàng, giữ đủ cột) (1, 3)
   student_id     name  age
0         101  Ulysses   13

BƯỚC 3 - [['name', 'age']]  (chọn cột) (1, 2)
      name  age
0  Ulysses   13

So với loc một bước:
      name  age
0  Ulysses   13
bằng nhau? True


In [18]:
def trace_loop(students):
    labels, rows = [], []
    for label, sid, name, age in zip(students.index, students["student_id"], students["name"], students["age"]):
        keep = sid == 101
        print(f"nhãn {label}: student_id={sid:>3}, name={name:<8}, age={age:>2} -> {'GIỮ' if keep else 'bỏ'}")
        if keep:
            labels.append(label)
            rows.append((name, age))
    out = pd.DataFrame(rows, index=labels, columns=["name", "age"])
    print(out)
    return out

_ = trace_loop(students)

nhãn 0: student_id=101, name=Ulysses , age=13 -> GIỮ
nhãn 1: student_id= 53, name=William , age=10 -> bỏ
nhãn 2: student_id=128, name=Henry   , age= 6 -> bỏ
nhãn 3: student_id=  3, name=Henry   , age=11 -> bỏ
      name  age
0  Ulysses   13


Bảng tóm tắt hình dạng (`shape`) qua các bước:

| bước | biểu thức | shape |
|---|---|---|
| 0 | `students` | `(4, 3)` |
| 1 | `mask` | `(4,)` (một cột `True/False`) |
| 2 | `students[mask]` | `(1, 3)` |
| 3 | `[["name", "age"]]` | `(1, 2)` |

## 9. Code nộp LeetCode

In [19]:
import pandas as pd

def selectData(students: pd.DataFrame) -> pd.DataFrame:
    return students.loc[students["student_id"] == 101, ["name", "age"]]

result = selectData(students)
print(result)
result

      name  age
0  Ulysses   13


,name,age
0,Ulysses,13


## 10. Vì sao thuật toán đúng

- **Lọc hàng đúng:** `students["student_id"] == 101` cho `True` đúng ở những hàng có `student_id` bằng `101` và `False` ở các hàng khác; `loc[mask, …]` giữ đúng các hàng `True`.
- **Thứ tự và nhãn hàng được giữ:** lọc không sắp xếp lại; mỗi hàng giữ nhãn gốc.
- **Chọn cột đúng:** danh sách `["name", "age"]` quyết định **tên và thứ tự** cột trong kết quả, đúng như đề (`name` rồi `age`); cột `student_id` bị bỏ.
- **Trả về DataFrame:** vì cột được đưa vào dạng **list** nên kết quả là DataFrame hai cột (dù chỉ có một hàng), không phải Series.
- **Không sửa đầu vào:** lọc / chọn tạo bảng mới; `students` giữ nguyên.
- **Nhiều hàng khớp:** nếu có nhiều hơn một `student_id == 101` thì tất cả đều được giữ; nếu không có hàng nào, kết quả là bảng 0 hàng 2 cột.

## 11. Phân tích Time / Space Complexity

Gọi `m` là số hàng, `c` số cột, `r` số hàng khớp.

| Cách | Time | Space |
|---|---|---|
| 1. Vòng lặp Python | `O(m)` (chạy trong Python, chậm) | `O(r)` |
| 2. `df[mask][cột]` | `O(m)` (vector hóa) | `O(m + r·c)` (bảng trung gian đủ cột) |
| 3. `loc[mask, cột]` | `O(m)` | `O(m + r·k)` với `k = 2` cột |
| 4. `query` | `O(m)` | `O(m + r·k)` |
| 5. `iloc` + vị trí | `O(m)` | `O(m + r·k)` |

Mọi cách phải **nhìn qua cả cột `student_id`** ít nhất một lần nên không thể tốt hơn `O(m)`. Khác biệt thực tế là hằng số (vector hóa nhanh hơn nhiều) và mức độ rõ ràng.

## 12. Bộ test

### 12.1. Ví dụ của đề và edge case

In [20]:
impls = {
    "loop": select_loop,
    "two_steps": select_two_steps,
    "loc": select_loc,
    "query": select_query,
    "iloc": select_iloc,
    "submit": selectData,
}

expected_example = pd.DataFrame({"name": ["Ulysses"], "age": [13]})
for key, fn in impls.items():
    pd.testing.assert_frame_equal(fn(students), expected_example)
print("PASS ví dụ của đề (mọi cách)")

PASS ví dụ của đề (mọi cách)


In [21]:
def make(ids, names=None, ages=None):
    n = len(ids)
    return pd.DataFrame({
        "student_id": ids,
        "name": names or [f"n{i}" for i in range(n)],
        "age": ages or [10 + i for i in range(n)],
    })

edge = {
    "hàng khớp ở cuối": make([5, 6, 101]),
    "hàng khớp ở giữa": make([5, 101, 6, 7]),
    "nhiều hàng khớp": make([101, 5, 101, 101]),
    "không hàng nào khớp": make([1, 2, 3]),
    "tất cả khớp": make([101, 101]),
    "chỉ một hàng, khớp": make([101]),
    "chỉ một hàng, không khớp": make([100]),
}
for name, df in edge.items():
    exp = df.loc[df["student_id"] == 101, ["name", "age"]]
    for key, fn in impls.items():
        got = fn(df)
        assert list(got.columns) == ["name", "age"], (name, key)
        assert got.shape == exp.shape, (name, key, got.shape, exp.shape)
        pd.testing.assert_frame_equal(got, exp, check_dtype=(len(exp) > 0), check_index_type=False)
        assert got.index.tolist() == exp.index.tolist(), (name, key)
    print("PASS", name, "->", exp.shape)

PASS hàng khớp ở cuối -> (1, 2)
PASS hàng khớp ở giữa -> (1, 2)
PASS nhiều hàng khớp -> (3, 2)
PASS không hàng nào khớp -> (0, 2)
PASS tất cả khớp -> (2, 2)
PASS chỉ một hàng, khớp -> (1, 2)
PASS chỉ một hàng, không khớp -> (0, 2)


(Với bảng kết quả rỗng, dtype của cách vòng lặp có thể là `object`, nên ta chỉ so dtype khi có hàng. Ta cũng bỏ qua **kiểu** của index (`RangeIndex` hay `Index` số nguyên) vì chỉ quan tâm **nhãn**, và nhãn được so riêng bằng `index.tolist()`.)

### 12.2. Nhãn (index) được giữ và không sửa đầu vào

In [22]:
df = make([7, 101, 8, 101, 9])
df.index = [10, 20, 30, 40, 50]                       # nhãn lộn xộn
for key, fn in impls.items():
    got = fn(df)
    assert got.index.tolist() == [20, 40], key        # giữ nhãn gốc của các hàng khớp
before = df.copy()
for key, fn in impls.items():
    fn(df)
pd.testing.assert_frame_equal(df, before)             # đầu vào không đổi
print("PASS nhãn gốc được giữ, đầu vào không bị sửa")

# Phản ví dụ: nếu quên chọn cột thì kết quả còn cột student_id
sai = df[df["student_id"] == 101]
assert list(sai.columns) == ["student_id", "name", "age"] and list(selectData(df).columns) == ["name", "age"]
print("PASS phản ví dụ quên chọn cột")

PASS nhãn gốc được giữ, đầu vào không bị sửa
PASS phản ví dụ quên chọn cột


### 12.3. Đối chiếu ngẫu nhiên (có seed)

Sinh bảng ngẫu nhiên `m ∈ [1, 60]` với `student_id ∈ [95, 105]` (để mã `101` xuất hiện nhiều hoặc không), tên và tuổi ngẫu nhiên, nhãn có thể xáo trộn; so mọi cách với đáp án dựng bằng vòng lặp thuần.

In [23]:
import random
random.seed(2880)
np.random.seed(2880)

def oracle(df):
    labels = [lab for lab, sid in zip(df.index, df["student_id"]) if sid == 101]
    return labels, [df.loc[lab, "name"] for lab in labels], [int(df.loc[lab, "age"]) for lab in labels]

for t in range(300):
    m = random.randint(1, 60)
    df = pd.DataFrame({
        "student_id": np.random.randint(95, 106, size=m),
        "name": [random.choice(["Ann", "Bao", "Chi", "Dung", "Henry"]) for _ in range(m)],
        "age": np.random.randint(5, 20, size=m),
    })
    if random.random() < 0.5:
        df.index = random.sample(range(500), m)
    labels, names, ages = oracle(df)
    for key, fn in impls.items():
        got = fn(df)
        assert got.index.tolist() == labels, (t, key)
        assert got["name"].tolist() == names and got["age"].tolist() == ages, (t, key)
        assert list(got.columns) == ["name", "age"]
print("300 bảng ngẫu nhiên: mọi cách khớp đáp án dựng bằng vòng lặp")

300 bảng ngẫu nhiên: mọi cách khớp đáp án dựng bằng vòng lặp


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Dùng `=` thay vì `==` (và vô tình **ghi đè** cột)

In [24]:
demo = students.copy()
demo["student_id"] = 101            # sai: đây là GÁN, làm mọi hàng có student_id = 101
print(demo)
print("đúng:", selectData(students)["name"].tolist())

   student_id     name  age
0         101  Ulysses   13
1         101  William   10
2         101    Henry    6
3         101    Henry   11
đúng: ['Ulysses']


**Vì sao sai:** `=` là phép **gán**; `==` là phép **so sánh**. Câu lệnh trên làm hỏng cả cột. (Trong ngoặc của `loc`/`[]`, viết `=` còn gây `SyntaxError`.) Nhớ: **lọc = `==`**.

### Lỗi 2: Dùng `and` thay vì `&`

In [25]:
try:
    students[(students["student_id"] == 101) and (students["age"] > 5)]     # sai
except ValueError as e:
    print("ValueError:", str(e)[:70])
ok = students[(students["student_id"] == 101) & (students["age"] > 5)]
print("đúng với & và ngoặc đơn:", ok.shape)

ValueError: The truth value of a Series is ambiguous. Use a.empty, a.bool(), a.ite
đúng với & và ngoặc đơn: (1, 3)


**Vì sao sai:** `and` cần một giá trị đúng/sai cho **cả cột**, nhưng cột có nhiều giá trị nên pandas báo `ValueError`. Với mask dùng `&`, `|`, `~` và bọc mỗi vế trong ngoặc đơn.

### Lỗi 3: `df["name", "age"]` (thiếu ngoặc vuông của list)

In [26]:
try:
    students["name", "age"]            # sai: coi ("name", "age") là MỘT khóa
except KeyError as e:
    print("KeyError:", e)
print(students[["name", "age"]].shape)     # đúng: hai ngoặc vuông

KeyError: ('name', 'age')
(4, 2)


**Vì sao sai:** `df["a", "b"]` truyền một **tuple** làm khóa. Muốn chọn nhiều cột phải đưa vào một **list**: `df[["a", "b"]]`.

### Lỗi 4: Quên chọn cột → còn cột `student_id`

In [27]:
sai = students[students["student_id"] == 101]
print("sai  :", sai.columns.tolist())                 # còn student_id
print("đúng :", selectData(students).columns.tolist())   # chỉ name, age

sai  : ['student_id', 'name', 'age']
đúng : ['name', 'age']


**Vì sao sai:** đề yêu cầu chỉ hiển thị `name` và `age`. Bảng trả về thừa cột sẽ không khớp.

### Lỗi 5: Chọn cột bằng chuỗi → ra Series chứ không phải DataFrame

In [28]:
sai = students.loc[students["student_id"] == 101, "name"]          # chuỗi -> Series
print(type(sai).__name__, sai.tolist())
dung = students.loc[students["student_id"] == 101, ["name", "age"]]   # list -> DataFrame
print(type(dung).__name__, dung.shape)

Series ['Ulysses']
DataFrame (1, 2)


**Vì sao sai:** chọn **một** cột bằng chuỗi trả về Series. Bài này cần DataFrame hai cột, nên truyền **list** tên cột.

### Lỗi 6: Sửa trực tiếp bảng gốc để "lọc"

In [29]:
def sai_xoa(students):
    students.drop(students[students["student_id"] != 101].index, inplace=True)   # sai: sửa input
    return students[["name", "age"]]

demo = students.copy()
print(sai_xoa(demo).shape, "| bảng gốc của người gọi còn", demo.shape)
print("đúng: students vẫn còn", students.shape, "hàng/cột sau khi gọi selectData")

(1, 2) | bảng gốc của người gọi còn (1, 3)
đúng: students vẫn còn (4, 3) hàng/cột sau khi gọi selectData


**Vì sao sai:** hàm vừa trả kết quả vừa **phá hủy** dữ liệu đầu vào (`inplace=True`). Lọc nên tạo bảng **mới**.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Chọn hàng thỏa điều kiện + chỉ hiển thị vài cột" → `df.loc[điều_kiện, [cột...]]`.
- Điều kiện nhiều vế: `(A) & (B)`, `(A) | (B)`, `~(A)` — luôn có ngoặc đơn.
- Lọc theo danh sách giá trị: `df["x"].isin([...])`.
- Chọn **một** cột → chuỗi (ra Series); chọn **nhiều** cột → list (ra DataFrame).
- Kiểm tra kết quả bằng `result.shape` và `result.columns.tolist()` trước khi nộp.
- Câu điều kiện dài, nhiều tên cột → `df.query("...")` dễ đọc hơn.

## 15. Biến thể của bài

1. **Lọc theo điều kiện số:** `df.loc[df["age"] > 10, ["name", "age"]]`.
2. **Lọc theo nhiều giá trị:** `df[df["student_id"].isin([101, 3])]`.
3. **Lọc chuỗi:** `df[df["name"].str.startswith("H")]`.
4. **Chọn cột theo kiểu dữ liệu:** `df.select_dtypes("number")`.
5. **Đổi tên cột sau khi chọn:** `.rename(columns={"name": "student_name"})`.
6. Liên quan LeetCode 2879 *Display the First Three Rows*, 2882 *Drop Duplicate Rows* và các bài lọc / sắp xếp trong *30 Days of Pandas*.

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vòng lặp thuần | duyệt từng hàng, dựng lại bảng | `O(m)` (Python, chậm) | `O(r)` | hiểu bản chất; tránh trên dữ liệu lớn |
| 2. `df[mask][cột]` | lọc hàng rồi chọn cột | `O(m)` | `O(m + r·c)` | dễ đọc, hai bước |
| 3. `loc[mask, cột]` | một bước | `O(m)` | `O(m + r·k)` | **cách nộp bài** |
| 4. `query(...)[cột]` | điều kiện dạng chuỗi | `O(m)` | `O(m + r·k)` | điều kiện phức tạp, nhiều biến |
| 5. `iloc` + vị trí | chọn theo vị trí số | `O(m)` | `O(m + r·k)` | cần làm việc theo vị trí / mảng |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `(students["age"] > 10).tolist()`, `students[students["age"] > 10].shape`, `type(students["name"]).__name__`, `type(students[["name"]]).__name__`.

**Bài 2 (tính tay).** Với bảng ví dụ ở mục 2, viết kết quả (hàng và cột) của `students.loc[students["name"] == "Henry", ["student_id", "age"]]`.

**Bài 3 (điền TODO).** Hoàn thành `select_adults_todo(students)`: các hàng có `age >= 10` và chỉ giữ cột `name`, `age`.

In [30]:
def select_adults_todo(students):
    # TODO 1: mask = students["age"] >= 10
    # TODO 2: return students.loc[mask, ["name", "age"]]
    pass

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi.

**Bài 4 (tự viết lại).** Viết `select_data_query(students)` bằng `query` mà không nhìn code mẫu. Kiểm tra với ví dụ của đề.

**Bài 5 (biến thể / mini challenge).** (a) Viết `select_by_ids(students, ids)`: giữ các hàng có `student_id` thuộc danh sách `ids`, chỉ cột `name`, `age`. (b) Viết `select_henry_over(students, age_min)`: các hàng có `name == "Henry"` **và** `age > age_min`, chỉ hai cột `student_id` và `age`.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `[True, False, False, True]`; `(2, 3)`; `Series`; `DataFrame`. Bài 2: tìm các hàng có `name = Henry` rồi chọn hai cột được yêu cầu.
- **Mức 2 (vừa):** Bài 2: hai hàng nhãn `2` và `3`: `(128, 6)` và `(3, 11)`. Bài 3: hai dòng đã ghi trong comment. Bài 4: `students.query("student_id == 101")[["name", "age"]]`.
- **Mức 3 (sâu):** Bài 5(a): `isin(ids)`. Bài 5(b): hai vế `(students["name"] == "Henry") & (students["age"] > age_min)`; nhớ ngoặc đơn, cột là list `["student_id", "age"]`.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [31]:
# Bài 1
assert (students["age"] > 10).tolist() == [True, False, False, True]
assert students[students["age"] > 10].shape == (2, 3)
assert type(students["name"]).__name__ == "Series"
assert type(students[["name"]]).__name__ == "DataFrame"

# Bài 2
b2 = students.loc[students["name"] == "Henry", ["student_id", "age"]]
assert b2.index.tolist() == [2, 3]
assert b2["student_id"].tolist() == [128, 3] and b2["age"].tolist() == [6, 11]

# Bài 3
def select_adults_todo_ans(students):
    mask = students["age"] >= 10
    return students.loc[mask, ["name", "age"]]

b3 = select_adults_todo_ans(students)
assert b3["name"].tolist() == ["Ulysses", "William", "Henry"] and b3.index.tolist() == [0, 1, 3]

# Bài 4
def select_data_query(students):
    return students.query("student_id == 101")[["name", "age"]]

pd.testing.assert_frame_equal(select_data_query(students), expected_example)

# Bài 5(a)
def select_by_ids(students, ids):
    return students.loc[students["student_id"].isin(ids), ["name", "age"]]

b5a = select_by_ids(students, [101, 3])
assert b5a["name"].tolist() == ["Ulysses", "Henry"] and b5a["age"].tolist() == [13, 11]

# Bài 5(b)
def select_henry_over(students, age_min):
    mask = (students["name"] == "Henry") & (students["age"] > age_min)
    return students.loc[mask, ["student_id", "age"]]

b5b = select_henry_over(students, 8)
assert b5b["student_id"].tolist() == [3] and b5b["age"].tolist() == [11]
assert len(select_henry_over(students, 20)) == 0
print("Tất cả đáp án đúng")

Tất cả đáp án đúng


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (lọc hàng + chọn cột)

```python
import pandas as pd

def selectData(students: pd.DataFrame) -> pd.DataFrame:
    return students.loc[students["student_id"] == 101, ["name", "age"]]
```

**Checklist**

- [ ] So sánh bằng `==` (không phải `=`), nhiều điều kiện dùng `&` / `|` và có ngoặc đơn?
- [ ] Danh sách cột là **list** (`["name", "age"]`) để kết quả là DataFrame?
- [ ] Đúng thứ tự cột `name`, `age` và không còn cột `student_id`?
- [ ] Không sửa bảng đầu vào (không `inplace=True`)?
- [ ] Đã thử trường hợp không có hàng nào khớp, nhiều hàng khớp, index lộn xộn?
- [ ] Skeleton đúng `def selectData(students: pd.DataFrame) -> pd.DataFrame` (đối chiếu lại với đề gốc nếu lệch)?